# Tests

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import config
from tqdm.auto import tqdm
from peft import PeftModel
import os
import time
import json
import re

# Import de tes modules personnalisés
from utils_data import load_reranking_data
from utils_reranking import RerankingCache, parse_vlm_ranking, calibrate_confidence_threshold
from utils_analysis import compute_autopsy, evaluate_and_save_results
from rerankers import Qwen2VLReranker

# ==========================================
# CONFIGURATION
# ==========================================
TOP_K_I2T = 5
TOP_K_T2I = 10
MODE_TEST = False 
NB_QUERIES_TEST = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Chargement du VLM (Modèle de base)
vlm = Qwen2VLReranker(model_id="Qwen/Qwen2-VL-7B-Instruct", device=device)
# ==========================================
# CHARGEMENT DES DONNÉES
# ==========================================
# 1. On charge la validation pour trouver le seuil optimal
val_dataset, _, S_t2i_val_stage1, S_i2t_val_stage1, targets_t2i_val_gpu, targets_i2t_val_gpu = load_reranking_data("val", device)

# 2. On charge le set de test pour l'évaluation finale
dataset, all_texts, S_t2i_stage1, S_i2t_stage1, targets_t2i_gpu, targets_i2t_gpu = load_reranking_data("test", device)

limit_i2t = NB_QUERIES_TEST if MODE_TEST else len(targets_i2t_gpu)
limit_t2i = NB_QUERIES_TEST if MODE_TEST else len(targets_t2i_gpu)

/home/marion/MIRAGE_TFE/env_marion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Chargement de Qwen/Qwen2-VL-7B-Instruct en Bfloat16 (Sans quantification)...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 730/730 [00:00<00:00, 1115.25it/s, Materializing param=model.visual.patch_embed.proj.weight]                         
Some parameters are on the meta device because they were offloaded to the cpu.


✅ Qwen2-VL chargé avec succès !

📦 Chargement des données pour le split : [VAL]...
  Chargé : ./data/flickr30k/index_sauvegardes/val_blip_img_index.bin (1014 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/val_blip_txt_index.bin (5070 vecteurs)
🔗 Calcul des matrices de similarité fusionnées...
✅ Données prêtes !

📦 Chargement des données pour le split : [TEST]...
  Chargé : ./data/flickr30k/index_sauvegardes/test_blip_img_index.bin (1000 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/test_blip_txt_index.bin (5000 vecteurs)
🔗 Calcul des matrices de similarité fusionnées...
✅ Données prêtes !


## I2T

In [ ]:
# ## Reranking I2T (SELF-CRITIQUE SOTA - 2ème Passe)

# %%
print("\n" + "="*50)
print("🚀 DÉMARRAGE I2T (SELF-CRITIQUE - 2ème Passe)")
print("="*50)

# 1. Chargement de tes prédictions initiales du VLM (Le fameux 0.971)
if not os.path.exists(config.CACHE_I2T_FILE):
    raise FileNotFoundError("Le cache initial est introuvable ! Lancez la première passe d'abord.")
    
with open(config.CACHE_I2T_FILE, 'r') as f:
    initial_vlm_predictions = json.load(f)

# 2. Création d'un NOUVEAU cache pour la critique (pour ne rien écraser)
CACHE_CRITIQUE_FILE = config.CACHE_I2T_FILE.replace('.json', '_critique.json')
cache_critique = RerankingCache(CACHE_CRITIQUE_FILE)

sorted_idx_i2t_base = torch.argsort(S_i2t_stage1, dim=1, descending=True)
sorted_idx_i2t_final = sorted_idx_i2t_base.clone()

# Sécurité LoRA
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    vlm.model.disable_adapters()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

for q_idx in tqdm(range(limit_i2t), desc="Inférence I2T Self-Critique"):
    top_k_idx = sorted_idx_i2t_base[q_idx, :TOP_K_I2T].tolist()
    
    # On vérifie si la critique a déjà été faite
    cached = cache_critique.get(q_idx)
    if cached:
        sorted_idx_i2t_final[q_idx, :TOP_K_I2T] = torch.tensor(cached, device=device)
        continue
        
    # On récupère l'ordre initial que le VLM avait choisi lors de la passe 1
    ordre_initial = initial_vlm_predictions[str(q_idx)]
    choix_top_1 = ordre_initial[0]
    
    image_query = dataset[q_idx]['image']
    textes_candidats = "\n".join([f"ID {idx}: {all_texts[idx]}" for idx in top_k_idx])
    
    # 🕵️‍♂️ LE PROMPT DE SELF-CRITIQUE
    prompt = f"""You are an expert image-to-text alignment evaluator.
    Here is a query image and {TOP_K_I2T} candidate descriptions with their unique IDs:
    
    {textes_candidats}

    Previously, you analyzed this image and ranked ID {choix_top_1} as the absolute best match.
    
    Your task: CRITICALLY RE-EVALUATE your choice.
    Look at the image very closely. Does the description in ID {choix_top_1} contain ANY subtle hallucination? (e.g., mentioning a color, an action, an object, or a background detail that is NOT visually present).
    
    - If ID {choix_top_1} is 100% flawless, you MUST keep it at the top.
    - If ID {choix_top_1} contains even a tiny hallucination, you MUST demote it and promote the true perfect match to the top.
    
    Think step-by-step to justify if your previous choice was correct or flawed.
    After your analysis, output the final ranking.
    
    Format:
    Final Ranking: [ID1, ID2, ID3, ID4, ID5]"""
    
    t0_call = time.time()
    
    response_text = vlm.generate_response(prompt, [image_query])
    
    t1_call = time.time()
    temps_vlm_reel += (t1_call - t0_call)
    appels_vlm_reels += 1

    # Parsing
    new_order = parse_vlm_ranking(response_text, top_k_idx)
    
    # Fallback si le parsing échoue ou renvoie une liste vide, on garde le 1er choix du VLM
    if len(new_order) == 0:
        new_order = ordre_initial
        
    sorted_idx_i2t_final[q_idx, :TOP_K_I2T] = torch.tensor(new_order, device=device)
    cache_critique.set(q_idx, new_order)
    
    if appels_vlm_reels % 5 == 0: 
        cache_critique.save()

cache_critique.save()

temps_total_vlm_i2t = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan I2T ---
evaluate_and_save_results(
    sorted_idx_i2t_base[:limit_i2t], 
    sorted_idx_i2t_final[:limit_i2t], 
    targets_i2t_gpu[:limit_i2t], 
    is_i2t=True, 
    csv_path=config.metriques_i2t_csv.replace('.csv', '_critique.csv'), 
    md_path=config.metriques_i2t_md.replace('.md', '_critique.md'),
    temps_vlm=temps_total_vlm_i2t
)

_ = compute_autopsy(sorted_idx_i2t_base, sorted_idx_i2t_final, targets_i2t_gpu[:limit_i2t], is_i2t=True, mask_vlm_called=[True]*limit_i2t)


🚀 DÉMARRAGE I2T (SELF-CRITIQUE - 2ème Passe)


Inférence I2T Self-Critique: 100%|██████████| 1000/1000 [00:02<00:00, 361.25it/s]



COMPARAISON FINALE : LATE FUSION vs VLM (I2T)
                                      R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)           0.960 0.999 1.000 0.883 0.948      0.000
2. Après (MIRAGE + VLM (Zero-Shot)) 0.967 0.999 1.000 0.878 0.946   8330.403

✅ Résultats globaux (avec temps) sauvegardés dans resultats_i2t_critique.csv et .md

--- BILAN COMPTABLE I2T ---
Total requêtes analysées : 1000
✅ Sauvetages (Faux -> Vrai) : 19
⚠️ Sabotages (Vrai -> Faux) : 12
❌ Doubles Échecs (Faux -> Faux) : 21
🆗 Confirmations (Vrai -> Vrai) : 948
📈 GAIN NET DE R@1 : 7 requêtes (soit +0.70%)


## T2I

### Pointwise Prompt Ensembling (Sans cascade)

In [ ]:
print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (POINTWISE SOTA + PROMPT ENSEMBLING)")
print("="*50)

# 1. Désactivation du LoRA
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")
    vlm.model.disable_adapters()
else:
    print("✅ Le modèle est bien en mode de base (Zero-Shot).")

# 2. Initialisation d'un nouveau cache pour l'Ensembling
CACHE_ENSEMBLE_FILE = config.CACHE_T2I_FILE.replace('.json', '_ensemble.json')
cache_t2i = RerankingCache(CACHE_ENSEMBLE_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0
ALPHA = 0.5 

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Prompt Ensembling"):
    top_5 = sorted_idx_t2i_base[q_idx, :TOP_K_T2I].tolist()
    
    # Vérification du cache
    cached = cache_t2i.get(q_idx)
    if cached:
        new_top_5 = cached
    else:
        images_pil = [dataset[idx]['image'] for idx in top_5]
        requete = all_texts[q_idx]
        
        # 📝 LES 3 PROMPTS SOTA
        prompt_1 = f"Does the following image exactly match this description: '{requete}'? Answer strictly with 'Yes' or 'No'."
        prompt_2 = f"Look closely at the fine details and the background. Is this a perfect visual representation of: '{requete}'? Answer strictly with 'Yes' or 'No'."
        prompt_3 = f"Is the description '{requete}' 100% factually correct for every element visible in this image? Answer strictly with 'Yes' or 'No'."
        
        t0_call = time.time()
        
        # On score les 5 images pour chaque prompt (3 appels successifs au batch)
        scores_p1 = vlm.score_image_pointwise_batch([prompt_1] * 5, images_pil)
        scores_p2 = vlm.score_image_pointwise_batch([prompt_2] * 5, images_pil)
        scores_p3 = vlm.score_image_pointwise_batch([prompt_3] * 5, images_pil)
        
        # On fait la moyenne des 3 probabilités VLM pour chaque image
        scores_vlm_moyens = [(s1 + s2 + s3) / 3.0 for s1, s2, s3 in zip(scores_p1, scores_p2, scores_p3)]
        
        t1_call = time.time()

        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1 

        # --- LATE FUSION ---
        scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_5]
        
        # Normalisation Min-Max
        max_s = max(scores_mirage_bruts)
        min_s = min(scores_mirage_bruts)
        if max_s > min_s:
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) for s in scores_mirage_bruts]
        else:
            scores_mirage_norm = [1.0] * 5

        # Fusion mathématique (Alpha * VLM_Moyen + (1-Alpha) * MIRAGE)
        final_scores = []
        for i, idx in enumerate(top_5):
            score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm_moyens[i])
            final_scores.append((score_mixte, idx))
            
        # Nouveau tri
        final_scores.sort(key=lambda x: x[0], reverse=True)
        new_top_5 = [idx for score, idx in final_scores]
        
        # Mise en cache
        cache_t2i.set(q_idx, new_top_5)
        if appels_vlm_reels % 5 == 0: 
            cache_t2i.save()
            
    ordre_final = new_top_5 + top_5[TOP_K_T2I:] if len(top_5) > TOP_K_T2I else new_top_5 + sorted_idx_t2i_base[q_idx, TOP_K_T2I:].tolist()
    sorted_idx_t2i_final[q_idx, :TOP_K_T2I] = torch.tensor(new_top_5, device=device)

cache_t2i.save()

# --- Calcul du temps projeté ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan T2I ---
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_ensemble.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_ensemble.md'),
    temps_vlm=temps_total_vlm_t2i
)

# --- Autopsie T2I ---
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=[True]*limit_t2i)


🚀 DÉMARRAGE T2I (POINTWISE SOTA + PROMPT ENSEMBLING)
✅ Le modèle est bien en mode de base (Zero-Shot).


Inférence T2I Prompt Ensembling:   0%|          | 0/5000 [00:00<?, ?it/s]

Inférence T2I Prompt Ensembling: 100%|██████████| 5000/5000 [00:16<00:00, 307.28it/s]



COMPARAISON FINALE : LATE FUSION vs VLM (T2I)
                                         R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)              0.868 0.974 0.988 0.916 0.936      0.000
2. Après (MIRAGE + VLM (Cascade LoRA)) 0.890 0.974 0.988 0.929 0.946  10049.284

✅ Résultats globaux (avec temps) sauvegardés dans resultats_t2i_ensemble.csv et .md

--- BILAN COMPTABLE T2I ---
Total requêtes analysées : 5000
✅ Sauvetages (Faux -> Vrai) : 137
⚠️ Sabotages (Vrai -> Faux) : 30
❌ Doubles Échecs (Faux -> Faux) : 521
🆗 Confirmations (Vrai -> Vrai) : 4312
📈 GAIN NET DE R@1 : 107 requêtes (soit +2.14%)


### Pointwise Top 5 (Sans cascade, Late Fusion α)

In [ ]:

print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (FULL POINTWISE SOTA - Sans Cascade)")
print("="*50)

# 1. Désactivation du LoRA (Le Pointwise fonctionne mieux sur le modèle de base non biaisé)
if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pointwise Zero-Shot...")
    vlm.model.disable_adapters()
else:
    print("✅ Le modèle est bien en mode de base (Zero-Shot).")

# 2. Initialisation
cache_t2i = RerankingCache(config.CACHE_T2I_FILE)
sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

# ⚖️ Poids de la fusion (0.5 = 50% MIRAGE / 50% VLM)
ALPHA = 0.5 

# Comme on envoie TOUT, le mask est 100% True
mask_vlm_called = [True] * limit_t2i

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Full Pointwise"):
    top_5 = sorted_idx_t2i_base[q_idx, :TOP_K_T2I].tolist()
    
    # Vérification du cache
    cached = cache_t2i.get(q_idx)
    if cached:
        new_top_5 = cached
    else:
        images_pil = [dataset[idx]['image'] for idx in top_5]
        requete = all_texts[q_idx]
        
        # ⚡ VERSION BATCHÉE T2I ⚡
        prompt = f"Does the following image exactly and perfectly match this description: '{requete}'? Look at every tiny detail. Answer strictly with 'Yes' or 'No'."
        prompts_batch = [prompt] * 5
        
        t0_call = time.time()
        
        # On score les 5 images D'UN COUP avec le même texte
        scores_vlm = vlm.score_image_pointwise_batch(prompts_batch, images_pil)
        
        t1_call = time.time()

        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1 # On compte ça comme 1 requête complète traitée

        # --- LATE FUSION ---
        # 1. On récupère les scores bruts de MIRAGE pour le top 5
        scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_5]
        
        # 2. Normalisation Min-Max locale des scores MIRAGE pour qu'ils soient entre 0 et 1
        max_s = max(scores_mirage_bruts)
        min_s = min(scores_mirage_bruts)
        if max_s > min_s:
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) for s in scores_mirage_bruts]
        else:
            scores_mirage_norm = [1.0] * 5

        # 3. Fusion mathématique (Alpha * VLM + (1-Alpha) * MIRAGE)
        final_scores = []
        for i, idx in enumerate(top_5):
            score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm[i])
            final_scores.append((score_mixte, idx))
            
        # 4. Nouveau tri basé sur le score fusionné
        final_scores.sort(key=lambda x: x[0], reverse=True)
        new_top_5 = [idx for score, idx in final_scores]
        
        # Mise en cache
        cache_t2i.set(q_idx, new_top_5)
        if appels_vlm_reels % 5 == 0: 
            cache_t2i.save()
            
    ordre_final = new_top_5 + top_5[TOP_K_T2I:] if len(top_5) > TOP_K_T2I else new_top_5 + sorted_idx_t2i_base[q_idx, TOP_K_T2I:].tolist()
    sorted_idx_t2i_final[q_idx, :TOP_K_T2I] = torch.tensor(new_top_5, device=device)

cache_t2i.save()

# --- Calcul du temps projeté ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# --- Bilan T2I (Métriques et Sauvegarde) ---
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv, 
    md_path=config.metriques_t2i_md,
    temps_vlm=temps_total_vlm_t2i
)

# --- Autopsie T2I (Sauvetages/Sabotages) ---
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)


🚀 DÉMARRAGE T2I (FULL POINTWISE SOTA - Sans Cascade)
✅ Le modèle est bien en mode de base (Zero-Shot).


Inférence T2I Full Pointwise: 100%|██████████| 5000/5000 [00:16<00:00, 305.67it/s]



COMPARAISON FINALE : LATE FUSION vs VLM (T2I)
                                         R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)              0.868 0.974 0.988 0.916 0.936      0.049
2. Après (MIRAGE + VLM (Cascade LoRA)) 0.891 0.974 0.988 0.929 0.946    722.739

✅ Résultats globaux (avec temps) sauvegardés dans resultats_t2i.csv et .md

--- BILAN COMPTABLE T2I ---
Total requêtes analysées : 5000
✅ Sauvetages (Faux -> Vrai) : 142
⚠️ Sabotages (Vrai -> Faux) : 31
❌ Doubles Échecs (Faux -> Faux) : 516
🆗 Confirmations (Vrai -> Vrai) : 4311
📈 GAIN NET DE R@1 : 111 requêtes (soit +2.22%)


### Listwise LoRA (Juge Final sur Top 5)

In [ ]:
# Tressssss long
'''
print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I ÉTAPE 3 : JUGE FINAL LISTWISE (LoRA)")
print("="*50)

# 1. On utilise le tenseur final de l'étape précédente (Top 10) comme nouveau point de départ
sorted_idx_t2i_stage2 = sorted_idx_t2i_final.clone()
sorted_idx_t2i_stage3 = sorted_idx_t2i_stage2.clone()

# 2. ⚙️ CHARGEMENT DU MODÈLE ELITE (LoRA)

if os.path.exists(config.LORA_OUTPUT_DIR):
    print(f"⚙️ Injection des poids LoRA ELITE CoT pour l'évaluation Listwise...")
    if not (getattr(vlm.model, "_hf_peft_config_loaded", False) or isinstance(vlm.model, PeftModel)):
        old_no_split = getattr(vlm.model, "_no_split_modules", None)
        vlm.model._no_split_modules = None
        vlm.model = PeftModel.from_pretrained(vlm.model, config.LORA_OUTPUT_DIR)
        vlm.model._no_split_modules = old_no_split
    vlm.model.eval()
    print("✅ Modèle Listwise prêt !")
else:
    print("⚠️ Attention : Dossier LoRA introuvable, le modèle tournera en Zero-Shot.")

# 3. Initialisation du cache pour cette 3ème étape
CACHE_STAGE3_FILE = config.CACHE_T2I_FILE.replace('.json', '_stage3_listwise.json')
cache_stage3 = RerankingCache(CACHE_STAGE3_FILE)

temps_vlm_reel = 0.0
appels_vlm_reels = 0

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Stage 3 (Juge Final)"):
    # On ne prend QUE le Top 5 de la phase précédente (qui a déjà été pré-trié par le Pointwise)
    top_5 = sorted_idx_t2i_stage2[q_idx, :5].tolist()
    les_suivants = sorted_idx_t2i_stage2[q_idx, 5:].tolist()
    
    cached = cache_stage3.get(q_idx)
    if cached:
        new_top_5 = cached
    else:
        requete_texte = all_texts[q_idx]
        images_pil = [dataset[idx]['image'] for idx in top_5]
        
        # Mapping des positions (1 à 5) vers les vrais IDs d'images
        pos_to_id = {i+1: top_5[i] for i in range(5)}
        
        # 📝 PROMPT LISTWISE (Adapté pour comparer 5 images)
        prompt = (
            f"You are an expert visual verifier. These 5 images are pre-ranked by an AI for the query: '{requete_texte}'. "
            f"Image 1 is mathematically the most probable match. Your task is to verify this. "
            f"Compare the images directly. Do NOT demote Image 1 unless another image clearly matches the subtle details better. "
            f"Think step by step, penalize visual hallucinations, and output the Final Ranking."
            f"\nFormat:\nFinal Ranking: [PositionA, PositionB, PositionC, PositionD, PositionE]"
        )
        
        t0_call = time.time()
        response_text = vlm.generate_response(prompt, images_pil) 
        t1_call = time.time()
        
        temps_vlm_reel += (t1_call - t0_call)
        appels_vlm_reels += 1
        
        # --- PARSING ---
        match = re.search(r'Final Ranking:\s*(?:\[)?(.*?)(?:\])?$', response_text, re.IGNORECASE | re.MULTILINE)
        predicted_positions = None
        if match:
            nums = [int(n) for n in re.findall(r'\d+', match.group(1))]
            predicted_positions = []
            for x in nums:
                if 1 <= x <= 5 and x not in predicted_positions:
                    predicted_positions.append(x)
            missing = [i for i in range(1, 6) if i not in predicted_positions]
            predicted_positions.extend(missing)
        
        # Fallback de sécurité : Si le parsing échoue, on garde l'ordre de l'étape 2
        if predicted_positions is None or len(predicted_positions) != 5:
            new_top_5 = top_5
        else:
            # Reconversion des positions (1, 2, 3...) en IDs réels
            new_top_5 = [pos_to_id[pos] for pos in predicted_positions]
            
        # Mise en cache
        cache_stage3.set(q_idx, new_top_5)
        if appels_vlm_reels % 5 == 0:
            cache_stage3.save()
            
    # Mise à jour du tenseur avec le nouvel ordre
    ordre_final = new_top_5 + les_suivants
    sorted_idx_t2i_stage3[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

cache_stage3.save()

temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

# =====================================================================
# ÉVALUATION GLOBALE (Stage 1 vs Stage 3)
# =====================================================================
print("\n" + "="*80)
print("🏆 COMPARAISON FINALE : MIRAGE (Base) vs PIPELINE 3 ÉTAGES (Coarse-to-Fine)")
print("="*80)

# On compare l'état initial (MIRAGE pur) avec l'état final (Après Pointwise + Listwise)
evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_stage3[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_stage3_final.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_stage3_final.md'),
    temps_vlm=temps_total_vlm_t2i
)

# Autopsie globale : on veut voir combien d'erreurs de MIRAGE on a corrigées au total
_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_stage3, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=[True]*limit_t2i)
'''

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1285317/1862032654.py:2: SyntaxWarning: invalid escape sequence '\s'
  '''


'\nprint("\n" + "="*50)\nprint("🚀 DÉMARRAGE T2I ÉTAPE 3 : JUGE FINAL LISTWISE (LoRA)")\nprint("="*50)\n\n# 1. On utilise le tenseur final de l\'étape précédente (Top 10) comme nouveau point de départ\nsorted_idx_t2i_stage2 = sorted_idx_t2i_final.clone()\nsorted_idx_t2i_stage3 = sorted_idx_t2i_stage2.clone()\n\n# 2. ⚙️ CHARGEMENT DU MODÈLE ELITE (LoRA)\n\nif os.path.exists(config.LORA_OUTPUT_DIR):\n    print(f"⚙️ Injection des poids LoRA ELITE CoT pour l\'évaluation Listwise...")\n    if not (getattr(vlm.model, "_hf_peft_config_loaded", False) or isinstance(vlm.model, PeftModel)):\n        old_no_split = getattr(vlm.model, "_no_split_modules", None)\n        vlm.model._no_split_modules = None\n        vlm.model = PeftModel.from_pretrained(vlm.model, config.LORA_OUTPUT_DIR)\n        vlm.model._no_split_modules = old_no_split\n    vlm.model.eval()\n    print("✅ Modèle Listwise prêt !")\nelse:\n    print("⚠️ Attention : Dossier LoRA introuvable, le modèle tournera en Zero-Shot.")\n\n# 3. I

### Pointwise CoT (Avec Cascade & Late Fusion α)

In [ ]:
# ==========================================
# FONCTION DE PARSING POUR LE CoT (Score sur 100)
# ==========================================
def parse_cot_score(response_text):
    """Extrait la note finale (Score: X) générée par le VLM."""
    match = re.search(r"Score\s*:\s*(\d+)", response_text, re.IGNORECASE)
    if match:
        return float(match.group(1)) / 100.0 
    return 0.0 

print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (POINTWISE CoT + CASCADE LATE FUSION)")
print("="*50)

TOP_K_TEST = 10

if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA...")
    vlm.model.disable_adapters()

# 1. Calibrage du seuil de confiance
print("🎯 Calibrage du seuil de confiance...")
seuil_t2i = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=0.85)

CACHE_TOP10_COT_FILE = config.CACHE_T2I_FILE.replace('.json', '_top10_cot_cascade.json')
cache_t2i_cot = RerankingCache(CACHE_TOP10_COT_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0
mask_vlm_called = [False] * limit_t2i
requetes_sauvees = 0

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Pointwise CoT + Cascade"):
    top_10 = sorted_idx_t2i_base[q_idx, :TOP_K_TEST].tolist()
    
    # --- CASCADE RERANKING ---
    score_top1 = S_t2i_stage1[q_idx, top_10[0]].item()
    score_top2 = S_t2i_stage1[q_idx, top_10[1]].item()
    ecart_confiance = score_top1 - score_top2
    
    if ecart_confiance > seuil_t2i:
        new_top_10 = top_10
        requetes_sauvees += 1
    else:
        mask_vlm_called[q_idx] = True
        cached = cache_t2i_cot.get(q_idx)
        if cached:
            new_top_10 = cached
        else:
            requete = all_texts[q_idx]
            
            # PROMPT AVEC CHAIN OF THOUGHT
            prompt = f"""Analyze this image step-by-step against the description: '{requete}'. 
            Check every object, action, and attribute mentioned. List any missing or conflicting details. 
            Finally, rate the match on a scale from 0 to 100 on a new line exactly like this: 'Score: X'."""
            
            scores_vlm = []
            
            # Évaluation POINTWISE (1 image à la fois = très peu de VRAM)
            for idx in top_10:
                img = dataset[idx]['image']
                
                t0_call = time.time()
                response_text = vlm.generate_response(prompt, [img])
                t1_call = time.time()
                
                temps_vlm_reel += (t1_call - t0_call)
                appels_vlm_reels += 1
                
                score = parse_cot_score(response_text)
                scores_vlm.append(score)
            
            # --- LATE FUSION DOUCE ---
            # Au lieu d'éliminer cash (tournoi), on combine les certitudes
            scores_mirage_bruts = [S_t2i_stage1[q_idx, idx].item() for idx in top_10]
            max_s = max(scores_mirage_bruts)
            min_s = min(scores_mirage_bruts)
            scores_mirage_norm = [(s - min_s) / (max_s - min_s) if max_s > min_s else 1.0 for s in scores_mirage_bruts]

            # Alpha fixe (ou dynamique) : on donne 50% de poids à MIRAGE et 50% au VLM
            ALPHA = 0.5 
            final_scores = []
            for i, idx in enumerate(top_10):
                score_mixte = ((1 - ALPHA) * scores_mirage_norm[i]) + (ALPHA * scores_vlm[i])
                final_scores.append((score_mixte, idx))
                
            final_scores.sort(key=lambda x: x[0], reverse=True)
            new_top_10 = [idx for score, idx in final_scores]
            
            cache_t2i_cot.set(q_idx, new_top_10)
            if appels_vlm_reels % 30 == 0:
                cache_t2i_cot.save()
            
    sorted_idx_t2i_final[q_idx, :TOP_K_TEST] = torch.tensor(new_top_10, device=device)

cache_t2i_cot.save()

print(f"\n✅ Économie Cascade : {requetes_sauvees}/{limit_t2i} requêtes sans appel VLM !")

# --- Bilan ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_cot_cascade.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_cot_cascade.md'),
    temps_vlm=temps_total_vlm_t2i
)

_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)


🚀 DÉMARRAGE T2I (POINTWISE CoT + CASCADE LATE FUSION)
🎯 Calibrage du seuil de confiance...
📊 Seuil calibré pour attraper 85.0% des erreurs de validation : 0.02411


Inférence T2I Pointwise CoT + Cascade:   0%|          | 0/50 [00:00<?, ?it/s]

Inférence T2I Pointwise CoT + Cascade:  26%|██▌       | 13/50 [54:37<2:35:26, 252.08s/it]


KeyboardInterrupt: 

### Tournoi Pairwise (Avec Cascade)


In [ ]:
# ==========================================
# FONCTION DE PARSING POUR LE TOURNOI
# ==========================================
def parse_pairwise_winner(response_text):
    """Extrait le choix du VLM (Image A ou Image B)."""
    # On cherche une réponse claire 'A' ou 'B'
    match = re.search(r'\b(A|B)\b', response_text.upper())
    if match:
        return match.group(1)
    # En cas d'hallucination ou de refus, on garde le champion en titre (A par défaut)
    return 'A'

print("\n" + "="*50)
print("🚀 DÉMARRAGE T2I (PAIRWISE TOURNAMENT + CASCADE CONFIDENCE - TOP 10)")
print("="*50)

TOP_K_TEST = 10

if hasattr(vlm.model, "disable_adapters") and getattr(vlm.model, "_hf_peft_config_loaded", False):
    print("⚙️ Désactivation des adaptateurs LoRA pour test Pairwise Zero-Shot...")
    vlm.model.disable_adapters()

# 1. Calibrage du seuil sur le set de validation (On cible 85% des erreurs)
print("🎯 Calibrage du seuil de confiance...")
seuil_t2i = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=0.85)

# Nouveau cache pour l'approche Tournoi
CACHE_TOP10_PAIR_FILE = config.CACHE_T2I_FILE.replace('.json', '_top10_pairwise.json')
cache_t2i_pair = RerankingCache(CACHE_TOP10_PAIR_FILE)

sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
sorted_idx_t2i_final = sorted_idx_t2i_base.clone()

temps_vlm_reel = 0.0
appels_vlm_reels = 0

# Par défaut, on ne fait pas appel au VLM (on mettra True que si on a un doute)
mask_vlm_called = [False] * limit_t2i
requetes_sauvees = 0

for q_idx in tqdm(range(limit_t2i), desc="Inférence T2I Pairwise (Tournoi)"):
    top_10 = sorted_idx_t2i_base[q_idx, :TOP_K_TEST].tolist()
    
    # --- CASCADE RERANKING : Vérification de la confiance ---
    score_top1 = S_t2i_stage1[q_idx, top_10[0]].item()
    score_top2 = S_t2i_stage1[q_idx, top_10[1]].item()
    ecart_confiance = score_top1 - score_top2
    
    # Si l'écart est supérieur au seuil, on fait aveuglément confiance à la Phase 1
    if ecart_confiance > seuil_t2i:
        new_top_10 = top_10
        requetes_sauvees += 1
    else:
        # La Phase 1 hésite, on lance le VLM
        mask_vlm_called[q_idx] = True
        
        # Vérification du cache
        cached = cache_t2i_pair.get(q_idx)
        if cached:
            new_top_10 = cached
        else:
            requete = all_texts[q_idx]
            
            # Le champion initial est le Top 1 de MIRAGE
            champion_idx = top_10[0]
            champion_img = dataset[champion_idx]['image']
            
            # Le tournoi commence (du Top 2 au Top 10)
            for i in range(1, TOP_K_TEST):
                challenger_idx = top_10[i]
                challenger_img = dataset[challenger_idx]['image']
                
                prompt = f"""You are an expert evaluator. I am looking for the image that BEST matches this description: '{requete}'
                
                Image A is the first image. Image B is the second image.
                Compare both images carefully against every detail of the text. 
                Which image is a better match? Answer STRICTLY with a single letter: 'A' or 'B'."""
                
                t0_call = time.time()
                
                # On envoie les 2 images au VLM (A = champion, B = challenger)
                response_text = vlm.generate_response(prompt, [champion_img, challenger_img])
                
                t1_call = time.time()
                temps_vlm_reel += (t1_call - t0_call)
                appels_vlm_reels += 1
                
                winner = parse_pairwise_winner(response_text)
                
                # Si le challenger (B) gagne, il devient le nouveau champion
                if winner == 'B':
                    champion_idx = challenger_idx
                    champion_img = challenger_img
            
            # --- RECONSTRUCTION DU CLASSEMENT ---
            # Le champion prend la 1ère place. 
            # Les autres gardent leur ordre relatif dicté par MIRAGE (Phase 1).
            new_top_10 = [champion_idx]
            for idx in top_10:
                if idx != champion_idx:
                    new_top_10.append(idx)
                    
            # Mise en cache
            cache_t2i_pair.set(q_idx, new_top_10)
            if appels_vlm_reels % 45 == 0: # Sauvegarde régulière (toutes les ~5 requêtes complètes)
                cache_t2i_pair.save()
            
    # Mise à jour du tenseur final
    sorted_idx_t2i_final[q_idx, :TOP_K_TEST] = torch.tensor(new_top_10, device=device)

cache_t2i_pair.save()

print(f"\n✅ Économie Cascade : {requetes_sauvees}/{limit_t2i} requêtes ont été traitées sans appeler le VLM !")

# --- Bilan T2I ---
temps_total_vlm_t2i = temps_vlm_reel if appels_vlm_reels > 0 else 0.0

evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', '_top10_pairwise_cascade.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', '_top10_pairwise_cascade.md'),
    temps_vlm=temps_total_vlm_t2i
)

_ = compute_autopsy(sorted_idx_t2i_base, sorted_idx_t2i_final, targets_t2i_gpu[:limit_t2i], is_i2t=False, mask_vlm_called=mask_vlm_called)


🚀 DÉMARRAGE T2I (PAIRWISE TOURNAMENT + CASCADE CONFIDENCE - TOP 10)
🎯 Calibrage du seuil de confiance...
📊 Seuil calibré pour attraper 85.0% des erreurs de validation : 0.02411


Inférence T2I Pairwise (Tournoi): 100%|██████████| 50/50 [03:51<00:00,  4.63s/it]



✅ Économie Cascade : 41/50 requêtes ont été traitées sans appeler le VLM !

COMPARAISON FINALE : LATE FUSION vs VLM (T2I)
                                         R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)              0.960 0.960 1.000 0.965 0.973      0.000
2. Après (MIRAGE + VLM (Cascade LoRA)) 0.900 0.960 1.000 0.935 0.950    231.268

✅ Résultats globaux (avec temps) sauvegardés dans resultats_t2i_top10_pairwise_cascade.csv et .md

--- BILAN COMPTABLE T2I ---
Total requêtes analysées : 9
✅ Sauvetages (Faux -> Vrai) : 0
⚠️ Sabotages (Vrai -> Faux) : 3
❌ Doubles Échecs (Faux -> Faux) : 2
🆗 Confirmations (Vrai -> Vrai) : 4
📈 GAIN NET DE R@1 : -3 requêtes (soit -6.00%)


### Enregistrement des scores et test des fusions

In [ ]:
# Fichier de stockage des scores bruts
RAW_SCORES_FILE = os.path.join(config.RESULTS_DIR, "scores_bruts_t2i.json")

# On récupère le Top 10 de base
sorted_idx_t2i_base = torch.argsort(S_t2i_stage1, dim=1, descending=True)
scores_all_queries = {}

# Chargement du cache existant pour ne pas recalculer si vous avez déjà des résultats
if os.path.exists(RAW_SCORES_FILE):
    with open(RAW_SCORES_FILE, 'r') as f:
        scores_all_queries = json.load(f)

for q_idx in tqdm(range(limit_t2i), desc="Extraction des scores bruts"):
    str_qidx = str(q_idx)
    top_10 = sorted_idx_t2i_base[q_idx, :10].tolist()
    
    if str_qidx in scores_all_queries:
        continue

    # 1. Récupération des images et de la requête
    images_pil = [dataset[idx]['image'] for idx in top_10]
    requete = all_texts[q_idx]
    prompt = f"Does the following image exactly and perfectly match this description: '{requete}'? Answer strictly with 'Yes' or 'No'."
    
    # 2. Inférence VLM (Scoring Pointwise par Batch)
    # On coupe en deux batchs de 5 pour la VRAM
    scores_vlm_p1 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[:5])
    scores_vlm_p2 = vlm.score_image_pointwise_batch([prompt] * 5, images_pil[5:])
    scores_vlm = scores_vlm_p1 + scores_vlm_p2
    
    # 3. Récupération des scores MIRAGE bruts
    scores_mirage = [S_t2i_stage1[q_idx, idx].item() for idx in top_10]
    
    # Sauvegarde des données brutes
    scores_all_queries[str_qidx] = {
        "candidate_ids": top_10,
        "vlm_scores": scores_vlm,     # Probabilités [0, 1]
        "mirage_scores": scores_mirage # Scores de similarité phase 1
    }
    
    if q_idx % 10 == 0:
        with open(RAW_SCORES_FILE, 'w') as f:
            json.dump(scores_all_queries, f)

with open(RAW_SCORES_FILE, 'w') as f:
    json.dump(scores_all_queries, f)

print(f"✅ Scores bruts sauvegardés dans : {RAW_SCORES_FILE}")

Extraction des scores bruts:   0%|          | 0/5000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Extraction des scores bruts:  31%|███       | 1550/5000 [40:05<4:07:05,  4.30s/it]

In [ ]:
# %% [markdown]
# ## ÉTAPE 2 : LABORATOIRE DE FUSION (TABLEAU COMPARATIF COMPLET)
# %%
import json
import numpy as np
import torch
import time
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

def evaluate_fusion(scores_dict, targets_gpu, method="additive", alpha=0.5, rrf_k=60, top_k=10, cascade_threshold=None):
    """Calcule le R@1 en simulant dynamiquement le Top-K et la Cascade."""
    correct_count = 0
    total = len(scores_dict)
    reranked_results = {}
    mask_vlm_called = {} # Pour garder la trace des requêtes sauvées par la cascade
    
    for q_idx_str, data in scores_dict.items():
        q_idx = int(q_idx_str)
        target = targets_gpu[q_idx].item()
        
        ids = data["candidate_ids"][:top_k]
        vlm = np.array(data["vlm_scores"][:top_k])
        mirage = np.array(data["mirage_scores"][:top_k])
        
        # 🛡️ --- LOGIQUE DE CASCADE --- 🛡️
        ecart_confiance = mirage[0] - mirage[1] if len(mirage) > 1 else 0
        
        if cascade_threshold is not None and ecart_confiance > cascade_threshold:
            # MIRAGE est sûr de lui : on valide sans utiliser le VLM
            new_top_k = ids
            mask_vlm_called[q_idx] = False
        else:
            # MIRAGE hésite : on fait la fusion
            mask_vlm_called[q_idx] = True
            mirage_norm = (mirage - mirage.min()) / (mirage.max() - mirage.min() + 1e-8)
            
            if method == "additive":
                final_scores = (alpha * vlm) + ((1 - alpha) * mirage_norm)
            elif method == "multiplicative":
                final_scores = vlm * mirage_norm
            elif method == "maximum":
                final_scores = np.maximum(vlm, mirage_norm)
            elif method == "concatenation":
                final_scores = (vlm * 1000.0) + mirage_norm
            elif method == "rrf":
                rank_vlm = np.argsort(np.argsort(-vlm)) + 1
                rank_mirage = np.argsort(np.argsort(-mirage_norm)) + 1
                final_scores = (1.0 / (rrf_k + rank_vlm)) + (1.0 / (rrf_k + rank_mirage))
                
            best_indices = np.argsort(-final_scores) 
            new_top_k = [ids[i] for i in best_indices]
            
        reranked_results[q_idx] = new_top_k
        if new_top_k[0] == target:
            correct_count += 1
            
    return correct_count / total, reranked_results, mask_vlm_called

# ==========================================
# 1. PRÉPARATION DES DONNÉES ET DES TESTS
# ==========================================
with open(RAW_SCORES_FILE, 'r') as f:
    scores_bruts = json.load(f)

print("\n" + "="*70)
print("🧪 LABORATOIRE DE FUSION : GÉNÉRATION DU TABLEAU COMPARATIF")
print("="*70)

options_top_k = [5, 10]
methodes_simples = ["multiplicative", "maximum", "concatenation", "rrf"]

# 🎯 NOUVEAUTÉ : Grid Search sur le Seuil de Cascade
print("🎯 Calibrage des multiples seuils de cascade sur le set de validation...")
options_cascade = {"Sans Cascade": None}

# On teste plusieurs niveaux d'exigence (de très permissif à très strict)
recalls_a_tester = [0.70, 0.80, 0.85, 0.90, 0.95, 0.99, 1.00]

for tr in recalls_a_tester:
    # Utilise S_t2i_val_stage1 et targets_t2i_val_gpu (déjà chargés dans ton notebook)
    seuil = calibrate_confidence_threshold(S_t2i_val_stage1, targets_t2i_val_gpu, is_i2t=False, target_recall=tr)
    
    # On ajoute ce seuil au dictionnaire des cascades à tester
    nom_cascade = f"Avec (Recall={tr:.2f})"
    options_cascade[nom_cascade] = seuil

resultats_tableau = []
grand_gagnant = {"r1": 0, "nom": "", "dict": {}, "mask": {}}

# ==========================================
# 2. EXÉCUTION DE TOUTES LES COMBINAISONS
# ==========================================
# On désactive l'affichage de tqdm pour que le tableau s'affiche proprement à la fin
for top_k in options_top_k:
    for nom_cascade, val_cascade in options_cascade.items():
        
        # 1. Tests des méthodes simples
        for methode in methodes_simples:
            r1, res_dict, mask = evaluate_fusion(scores_bruts, targets_t2i_gpu, method=methode, top_k=top_k, cascade_threshold=val_cascade)
            req_sauvees = sum(not v for v in mask.values())
            
            resultats_tableau.append({
                "Profondeur": f"Top {top_k}",
                "Cascade": nom_cascade.split(" ")[0], # Juste "Sans" ou "Avec"
                "Stratégie de Fusion": methode.capitalize(),
                "R@1 (%)": r1 * 100,
                "VLM Évités": req_sauvees
            })
            
            if r1 > grand_gagnant["r1"]:
                grand_gagnant = {"r1": r1, "nom": f"{methode.capitalize()}_Top{top_k}_{nom_cascade.split(' ')[0]}", "dict": res_dict, "mask": mask}

        # 2. Test Additif avec Grid Search Intégré
        best_add_r1, best_alpha, best_add_dict, best_add_mask = 0, 0, {}, {}
        for a in np.linspace(0, 1, 21):
            r1, res_dict, mask = evaluate_fusion(scores_bruts, targets_t2i_gpu, method="additive", alpha=a, top_k=top_k, cascade_threshold=val_cascade)
            if r1 > best_add_r1:
                best_add_r1 = r1
                best_alpha = a
                best_add_dict = res_dict
                best_add_mask = mask
                
        req_sauvees_add = sum(not v for v in best_add_mask.values())
        resultats_tableau.append({
            "Profondeur": f"Top {top_k}",
            "Cascade": nom_cascade.split(" ")[0],
            "Stratégie de Fusion": f"Grid Search (Alpha={best_alpha:.2f})",
            "R@1 (%)": best_add_r1 * 100,
            "VLM Évités": req_sauvees_add
        })
        
        if best_add_r1 > grand_gagnant["r1"]:
            grand_gagnant = {"r1": best_add_r1, "nom": f"Grid_Search_A{best_alpha:.2f}_Top{top_k}_{nom_cascade.split(' ')[0]}", "dict": best_add_dict, "mask": best_add_mask}

# ==========================================
# 3. AFFICHAGE DU CLASSEMENT
# ==========================================
# Création du DataFrame Pandas et tri par les meilleurs scores R@1
df_resultats = pd.DataFrame(resultats_tableau)
df_resultats = df_resultats.sort_values(by=["R@1 (%)", "VLM Évités"], ascending=[False, False]).reset_index(drop=True)

print("\n🏆 CLASSEMENT DES STRATÉGIES DE RERANKING :\n")
display(df_resultats) # Affiche un beau tableau HTML dans Jupyter

print("\n" + "⭐ "*15)
print(f"LE GRAND GAGNANT ABSOLU EST : {grand_gagnant['nom']}")
print(f"R@1 : {grand_gagnant['r1']*100:.2f} %")
print("⭐ "*15 + "\n")

# ==========================================
# 4. RECONSTRUCTION ET AUTOPSIE DU GAGNANT
# ==========================================
print(f"⚙️ Sauvegarde et Autopsie de la meilleure configuration ({grand_gagnant['nom']})...")

sorted_idx_t2i_final = sorted_idx_t2i_base.clone()
for q_idx_str, new_top_k in grand_gagnant["dict"].items():
    q_idx = int(q_idx_str)
    ordre_final = new_top_k + sorted_idx_t2i_base[q_idx, len(new_top_k):].tolist()
    sorted_idx_t2i_final[q_idx, :len(ordre_final)] = torch.tensor(ordre_final, device=device)

# Reconstruction de la liste mask_vlm_called pour l'autopsie
mask_vlm_called_list = [False] * limit_t2i
for q_idx_str, was_called in grand_gagnant["mask"].items():
    mask_vlm_called_list[int(q_idx_str)] = was_called

# Nommage propre du fichier CSV
file_suffix = f"_{grand_gagnant['nom'].replace(' ', '_').replace('.', '').lower()}"

evaluate_and_save_results(
    sorted_idx_t2i_base[:limit_t2i], 
    sorted_idx_t2i_final[:limit_t2i], 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    csv_path=config.metriques_t2i_csv.replace('.csv', f'{file_suffix}.csv'), 
    md_path=config.metriques_t2i_md.replace('.md', f'{file_suffix}.md'),
    temps_vlm=0.0
)

_ = compute_autopsy(
    sorted_idx_t2i_base, 
    sorted_idx_t2i_final, 
    targets_t2i_gpu[:limit_t2i], 
    is_i2t=False, 
    mask_vlm_called=mask_vlm_called_list
)


🧪 LABORATOIRE DE FUSION : GÉNÉRATION DU TABLEAU COMPARATIF

🏆 CLASSEMENT DES STRATÉGIES DE RERANKING :



,Profondeur,Cascade,Stratégie de Fusion,R@1 (%),VLM Évités
0,Top 10,Avec,Grid Search (Alpha=0.65),90.4,591
1,Top 10,Sans,Grid Search (Alpha=0.60),90.3,0
2,Top 5,Avec,Grid Search (Alpha=0.75),90.2,591
3,Top 5,Avec,Multiplicative,90.0,591
4,Top 5,Sans,Grid Search (Alpha=0.65),90.0,0
5,Top 10,Avec,Multiplicative,89.9,591
6,Top 5,Sans,Multiplicative,89.9,0
7,Top 10,Sans,Multiplicative,89.8,0
8,Top 10,Avec,Rrf,89.4,591
9,Top 5,Avec,Rrf,89.2,591



⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ 
LE GRAND GAGNANT ABSOLU EST : Grid_Search_A0.65_Top10_Avec
R@1 : 90.40 %
⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ ⭐ 

⚙️ Sauvegarde et Autopsie de la meilleure configuration (Grid_Search_A0.65_Top10_Avec)...

COMPARAISON FINALE : LATE FUSION vs VLM (T2I)
                                         R@1   R@5  R@10   mAP  NDCG  Temps (s)
1. Avant (Stage 1 Fusion)              0.884 0.976 0.992 0.926 0.943      0.000
2. Après (MIRAGE + VLM (Cascade LoRA)) 0.904 0.985 0.992 0.938 0.953      0.000

✅ Résultats globaux (avec temps) sauvegardés dans resultats_t2i_grid_search_a065_top10_avec.csv et .md

--- BILAN COMPTABLE T2I ---
Total requêtes analysées : 409
✅ Sauvetages (Faux -> Vrai) : 35
⚠️ Sabotages (Vrai -> Faux) : 15
❌ Doubles Échecs (Faux -> Faux) : 79
🆗 Confirmations (Vrai -> Vrai) : 280
📈 GAIN NET DE R@1 : 20 requêtes (soit +2.00%)
